# Lab 03 Lab 03: Data Cleaning and Transformation Pipeline

Santiago Elí Jiménez Aguilar 

In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Lab 03 - Dataset"

su =SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 03:37:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkUtils: Lab 03 - Dataset>

In [2]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns_types = [("Order_ID", "long"),
                 ("Company", "string"),
                 ("City", "string"),
                 ("Customer_Age", "int"),
                 ("Order_Value", "float"),
                 ("Delivery_Time_Min", "float"),
                 ("Distance_Km", "float"),
                 ("Items_Count", "float"),
                 ("Product_Category", "string"),
                 ("Payment_Method", "string"),
                 ("Customer_Rating", "float"),
                 ("Discount_Applied", "float"),
                 ("Delivery_Partner_Rating", "float")
                 ]

qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/")

qcommerce_df.printSchema()

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



In [3]:
qcommerce_df.limit(5).toPandas()

,Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
0,1000001,Swiggy Instamart,Noida,46,702.337524,19.181999,11.97,12.0,Dairy,Wallet,2.1,1.0,3.2
1,1000002,Flipkart Minutes,Amritsar,56,1007.299988,19.643999,12.74,10.0,Snacks,Cash on Delivery,2.3,0.0,3.2
2,1000003,Flipkart Minutes,Mumbai,18,1211.660034,16.910000,4.85,NaN,Personal Care,Cash on Delivery,3.3,0.0,3.8
3,1000004,Swiggy Instamart,Delhi,23,1179.059204,5.864000,6.44,2.0,Dairy,Credit Card,5.0,1.0,5.0
4,1000005,Dunzo,Mumbai,44,586.025513,12.470000,2.45,13.0,Household,Wallet,3.7,0.0,4.8


In [4]:
from pyspark.sql.functions import count, when, isnull

qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])

qcommerce_null_count_df.toPandas()

,Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
0,0,0,52000,0,0,0,0,35000,0,0,47000,0,104137


In [5]:
qcommerce_null_count_df_2 = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df_2.limit(5).toPandas()

,Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
0,0,0,52000,0,0,0,0,35000,0,0,47000,0,104137


In [6]:
qcommerce_no_null = qcommerce_df.dropna()
print(f"size of data frame after removing nulls: {qcommerce_no_null.count()}")

[Stage 7:=============================>                             (1 + 1) / 2]

size of data frame after removing nulls: 780992


In [7]:
qcommerce_no_nulls_fillna = qcommerce_df.fillna({
    'City' : 'Unknown',
    'Items_Count' : 0.0,
    'Customer_Rating' : 0.0,
    'Delivery_Partner_Rating' : 0.0,
})
print(f"size of data frame after removing nulls: {qcommerce_no_nulls_fillna.count()}")

[Stage 10:=============================>                            (1 + 1) / 2]

size of data frame after removing nulls: 1000000


In [8]:
from pyspark.sql.functions import lit

qcommerce_no_nulls_fillna = qcommerce_no_nulls_fillna.withColumn("Code_Country", lit("IN"))
qcommerce_no_nulls_fillna.limit(5).toPandas()

,Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating,Code_Country
0,1000001,Swiggy Instamart,Noida,46,702.337524,19.181999,11.97,12.0,Dairy,Wallet,2.1,1.0,3.2,IN
1,1000002,Flipkart Minutes,Amritsar,56,1007.299988,19.643999,12.74,10.0,Snacks,Cash on Delivery,2.3,0.0,3.2,IN
2,1000003,Flipkart Minutes,Mumbai,18,1211.660034,16.910000,4.85,0.0,Personal Care,Cash on Delivery,3.3,0.0,3.8,IN
3,1000004,Swiggy Instamart,Delhi,23,1179.059204,5.864000,6.44,2.0,Dairy,Credit Card,5.0,1.0,5.0,IN
4,1000005,Dunzo,Mumbai,44,586.025513,12.470000,2.45,13.0,Household,Wallet,3.7,0.0,4.8,IN


In [9]:
from pyspark.sql.functions import col

qcommerce_tax_df = qcommerce_no_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.limit(5).toPandas()

,Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating,Code_Country,Paid_Tax
0,1000001,Swiggy Instamart,Noida,46,702.337524,19.181999,11.97,12.0,Dairy,Wallet,2.1,1.0,3.2,IN,112.374004
1,1000002,Flipkart Minutes,Amritsar,56,1007.299988,19.643999,12.74,10.0,Snacks,Cash on Delivery,2.3,0.0,3.2,IN,161.167998
2,1000003,Flipkart Minutes,Mumbai,18,1211.660034,16.910000,4.85,0.0,Personal Care,Cash on Delivery,3.3,0.0,3.8,IN,193.865605
3,1000004,Swiggy Instamart,Delhi,23,1179.059204,5.864000,6.44,2.0,Dairy,Credit Card,5.0,1.0,5.0,IN,188.649473
4,1000005,Dunzo,Mumbai,44,586.025513,12.470000,2.45,13.0,Household,Wallet,3.7,0.0,4.8,IN,93.764082


# Lab 03: Data Cleaning and Transformation Pipeline
Create a Data pipeline in a Jupyter Notebook to perform the following
transformations of the Quick Commerce dataset from Kaggle.

## Task 1: Delivery SLA classification + filtering
Create Delivery_SLA with withColumn + when:<br>
Delivery_Time_Min ≤ 15 → "FAST", 15--20 → "ON_TIME", > 20 → "LATE"<br>
filter only Delivery_SLA = "LATE" and orderBy Delivery_Time_Min (desc).<br>
Display: Order_ID, Company, City, Delivery_Time_Min, Delivery_SLA.

In [10]:
qcommerce_df_task1 = qcommerce_tax_df.withColumn("Delivery_SLA", 
    when(col("Delivery_Time_Min") <= 15, "FAST") \
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON-TIME") \
    .when(col("Delivery_Time_Min") > 20, "LATE")) \
    .filter(col("Delivery_SLA") == "LATE") \
    .orderBy(col("Delivery_Time_Min"), ascending=False)

qcommerce_df_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").toPandas()

,Order_ID,Company,City,Delivery_Time_Min,Delivery_SLA
0,1529779,Jio Mart,Haridwar,40.000000,LATE
1,1807126,Jio Mart,Haridwar,40.000000,LATE
2,1165764,Jio Mart,Haridwar,39.993999,LATE
3,1610720,Jio Mart,Haridwar,39.993999,LATE
4,1729503,Jio Mart,Haridwar,39.993999,LATE
...,...,...,...,...,...
259643,1965412,Big Basket,Mumbai,20.002001,LATE
259644,1970920,Amazon Now,Noida,20.002001,LATE
259645,1974566,Big Basket,Chennai,20.002001,LATE
259646,1980969,Amazon Now,Pune,20.002001,LATE


# Task 2: Discount impact (effective price + bucket + grouping)
Create Effective_Order_Value = Order_Value * (1 - Discount_Applied).<br>
Create Price_Bucket with when: <br>
    < 200 → "LOW", 200--500 → "MEDIUM", > 500 → "HIGH"<br>
groupBy(Price_Bucket) and compute count(*) and avg(Effective_Order_Value).<br>
orderBy average effective value (desc).

In [11]:
from pyspark.sql.functions import avg

qcommerce_df_task2 = qcommerce_df_task1.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied")))

qcommerce_df_task2 = qcommerce_df_task2.withColumn("Price_Bucket", 
    when(col("Effective_Order_Value") <= 200, "LOW") \
    .when((col("Effective_Order_Value") > 200) & (col("Effective_Order_Value") <= 500), "MEDIUM") \
    .when(col("Effective_Order_Value") > 500, "HIGH")) \
    .groupBy("Price_Bucket") \
    .agg(count("*").alias("Order_Count"), avg("Effective_Order_Value").alias("Avg_Effective_Order_Value")) \
    .orderBy(col("Avg_Effective_Order_Value"), ascending=False)

qcommerce_df_task2.toPandas()

,Price_Bucket,Order_Count,Avg_Effective_Order_Value
0,HIGH,56013,707.219837
1,MEDIUM,59824,353.491340
2,LOW,143811,25.364970


# Task 3: Customer segmentation (Age_Group) + category insights

Create Age_Group with withColumn + when: <br>
< 25 → "YOUNG", 25--44 → "ADULT", ≥ 45 → "SENIOR"<br>
filter invalid ages (nulls, < 0, or > 100).<br>
groupBy(Age_Group, Product_Category) and compute:<br>
orders = count(*), avg_order_value = avg(Order_Value)<br>
orderBy(Age_Group asc, orders desc) to find top categories per segment.

In [12]:
qcommerce_df_task3 = qcommerce_tax_df \
    .filter(col("Customer_Age").isNotNull() & (col("Customer_Age") >= 0) & (col("Customer_Age") <= 100)) \
    .withColumn("Age_Group", 
     when(col("Customer_Age") < 25, "YOUNG") \
    .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT") \
    .when(col("Customer_Age") >= 45, "SENIOR")) \
    .groupBy("Age_Group", "Product_Category") \
    .agg(count("*").alias("orders"), avg("Order_Value").alias("avg_order_value")) \
    .orderBy(col("Age_Group").asc(), col("orders").desc())

qcommerce_df_task3.toPandas()

,Age_Group,Product_Category,orders,avg_order_value
0,ADULT,Dairy,68512,573.426890
1,ADULT,Personal Care,68331,569.512672
2,ADULT,Groceries,68291,571.525099
3,ADULT,Household,68110,572.925915
4,ADULT,Snacks,68100,570.379797
5,ADULT,Beverages,67979,572.032988
6,ADULT,Fruits & Vegetables,67736,569.859360
7,SENIOR,Groceries,51030,572.259639
8,SENIOR,Dairy,51025,573.781117
9,SENIOR,Snacks,50924,572.687852


In [13]:
su.spark_context().stop()